# bangla-pi-uni-gemma — Unicode-preserving rerun arm — gemma-4-26b-a4b
DECISIONS D40/D41. **Secondary, post hoc arm**, not part of the pre-registered primary analysis.
AgentDojo 0.1.35 formats structured tool output with `yaml.safe_dump(allow_unicode=False)`, so in the stock
main grid the Bengali injection reached the agent as `\uXXXX` escapes in these episodes. This notebook reruns
**exactly those episodes** with `runners/unicode_patch.py` ON (`allow_unicode=True` for tool output +
`ensure_ascii=False` when replaying the agent's own tool-call arguments) and **nothing else changed**.

**Episodes (tier A, run):** 190 = {'bn_mt': 95, 'bn_llm': 95}. Tier B (escapes only outside the injected block or
non-ASCII tool args; 107 = {'en': 36, 'bn_mt': 18, 'bn_llm': 18, 'banglish_llm': 35}) is listed but **not run** unless `TIERS` is edited.
`en`/`banglish_llm` injected text is ASCII, so those cells are reused from the stock run.

**Model / serve (identical to the main grid):** `google/gemma-4-26B-A4B-it-qat-q4_0-gguf` → `gemma-4-26B_q4_0-it.gguf` @ `d1c082be9cf3` (SHA-256 asserted);
QAT Q4_0; 1 server tensor-split 1,1; --swa-full + thinking off (D20 primary variant); `-c 24576 -np 1 --jinja --temp 0 --seed 20261002`, llama.cpp `b11344`,
1 agent stream per server.

**Estimate:** 0.86 GPU-h (high 0.97), from the observed main-grid
durations of these very episodes (1 stream(s), observed setup
0.07 h). Deadline guard: no new episodes after 11 h.

**Self-contained:** cell 1 unpacks the embedded harness (tarball 114,391 B gz, 168 files,
sha256 `8569d18166a4093193d7818a709b586866f401b53ff5f953d5e6adf4d4f9d963`), verifies EMBED_MANIFEST, PAYLOAD_SHA256SUMS, and that the 162 stock harness files are
byte-identical to the main-grid embed `ee7b4195e694…` that produced the stock results.

**Outputs (/kaggle/working):** `runs/gemma_150_uni.jsonl`, `runs/traces_uni/…`, `gemma_150_uni_traces.zip`,
`runs_manifest_gemma_150_uni.json`, `server_*.log`,
final `UNI_RESULT …` line(s). Resume: attach this Output as an input and
Save & Run All again (skip-if-done).

Settings: Accelerator **GPU T4 x2**, Internet **on**, notebook **private**, title **`bangla-pi-uni-gemma`**, then
**Save Version → Save & Run All (Commit)**. Generated by `scripts/build_uni_rerun_notebook.py`; **not run**.

In [ ]:
# ===== 1. Unpack the embedded harness + verify PAYLOAD_SHA256SUMS (fails loudly) =====
# Self-contained: no Kaggle dataset or git clone. Built by scripts/build_main_grid_notebook.py.
import base64, hashlib, io, json, os, shutil, sys, tarfile, time
from pathlib import Path
SESSION_T0 = time.time()
STOCK_EMBED_SHA256 = 'ee7b4195e694977b3f4e060155d66ca373027cdc2dee98ee88750b5305169334'   # main-grid tarball that ran (D32-D37)
STOCK_MANIFEST_SHA256 = 'beb44a7dba688ffa8d1a2030c6451cd6c515d8d8c3caec8d26d657d06f898ba9'
CODE_DIR = '/kaggle/working/code'
EMBED_SHA256 = '8569d18166a4093193d7818a709b586866f401b53ff5f953d5e6adf4d4f9d963'
EMBED_BYTES = 114391
EMBED_N_FILES = 168
EMBED_B64 = '''
...
AA==
'''

_blob = base64.b64decode(''.join(EMBED_B64.split()))
EMBED_SHA256_GOT = hashlib.sha256(_blob).hexdigest()
print(f'EMBED tarball: {len(_blob)} B, sha256 {EMBED_SHA256_GOT}', flush=True)
if len(_blob) != EMBED_BYTES or EMBED_SHA256_GOT != EMBED_SHA256:
    raise RuntimeError(f'EMBEDDED TARBALL CORRUPT: got {len(_blob)} B sha256 {EMBED_SHA256_GOT}, '
                       f'expected {EMBED_BYTES} B sha256 {EMBED_SHA256}. Re-import the notebook.')
if os.path.isdir(CODE_DIR):
    shutil.rmtree(CODE_DIR)          # always run exactly the embedded tree
os.makedirs(CODE_DIR)
with tarfile.open(fileobj=io.BytesIO(_blob), mode='r:gz') as _tf:
    _members = _tf.getmembers()
    for _m in _members:
        if not _m.isfile() or _m.name.startswith('/') or '..' in _m.name.split('/'):
            raise RuntimeError(f'unsafe tar member {_m.name!r}')
    if hasattr(tarfile, 'data_filter'):
        _tf.extractall(CODE_DIR, members=_members, filter='data')
    else:
        _tf.extractall(CODE_DIR, members=_members)
_root = Path(CODE_DIR)
_sha = lambda b: hashlib.sha256(b).hexdigest()
_bad = []

# (a) every embedded file vs EMBED_MANIFEST.json, and nothing extra
EMBED_MANIFEST = json.loads((_root / 'EMBED_MANIFEST.json').read_text(encoding='utf-8'))
for _rel, _meta in sorted(EMBED_MANIFEST['files'].items()):
    _p = _root / _rel
    _got = _sha(_p.read_bytes()) if _p.is_file() else 'MISSING'
    if _got != _meta['sha256']:
        _bad.append(f'manifest: {_rel} sha256 {_got[:16]} != {_meta["sha256"][:16]}')
_extra = sorted(str(p.relative_to(_root)) for p in _root.rglob('*') if p.is_file()
                and str(p.relative_to(_root)) not in EMBED_MANIFEST['files'] and p.name != 'EMBED_MANIFEST.json')
if _extra:
    _bad.append(f'manifest: unexpected files {_extra[:5]}')

# (b) PAYLOAD_SHA256SUMS.txt: every frozen payload file + CANONICAL goal-map hash of payloads.json
_C = _root / 'data' / 'conditions'
SUMS_TEXT = (_C / 'PAYLOAD_SHA256SUMS.txt').read_text(encoding='utf-8')
PAYLOAD_SUMS_SHA256 = _sha(SUMS_TEXT.encode('utf-8'))
_pay = json.loads((_root / 'data' / 'payloads.json').read_text(encoding='utf-8'))
_n_files = _n_canon = 0
_sum_conds = set()
for _line in SUMS_TEXT.splitlines():
    if not _line.strip() or _line.startswith('#'):
        continue
    _digest, _name = _line.split('  ', 1)
    if _name.startswith('CANONICAL_GOALS_JSON('):
        _cond = _name[len('CANONICAL_GOALS_JSON('):-1]
        _got = _sha(json.dumps(_pay['conditions'][_cond]['goals'], ensure_ascii=False, sort_keys=True,
                               separators=(',', ':')).encode('utf-8'))
        _n_canon += 1
        _sum_conds.add(_cond)
    else:
        _p = _C / _name
        _got = _sha(_p.read_bytes()) if _p.is_file() else 'MISSING'
        _n_files += 1
    if _got != _digest:
        _bad.append(f'PAYLOAD_SHA256SUMS: {_name} sha256 {_got[:16]} != {_digest[:16]}')

# (c) the texts the attack loads at run time (payloads.json) == the sha-verified files
def _body(_t):
    _ls = _t.split('\n'); _i = 0
    while _i < len(_ls) and _ls[_i].startswith('#'):
        _i += 1
    return '\n'.join(_ls[_i:]).strip()
_n_goal_files = 0
for _cond in sorted(_sum_conds):
    for _p in sorted((_C / _cond).glob('*__injection_task_*.txt')):
        _key = _p.stem.replace('__', '/')
        if _body(_p.read_text(encoding='utf-8')) != str(_pay['conditions'][_cond]['goals'].get(_key, '')).strip():
            _bad.append(f'payloads.json {_cond} {_key} != sha-verified file')
        _n_goal_files += 1
    _w = (_C / '_wrappers' / f'W1__{_cond}.txt').read_text(encoding='utf-8')
    if _body(_w) != str(_pay['wrappers']['W1'].get(_cond, '')).strip():
        _bad.append(f'payloads.json W1 wrapper {_cond} != sha-verified file')

if _bad:
    print('\n'.join(_bad), flush=True)
    raise RuntimeError(f'PAYLOAD/EMBED VERIFICATION FAILED ({len(_bad)} problems) - do not run the grid')
PAYLOAD_VERIFY = {'files': _n_files, 'canonical': _n_canon, 'goal_files_vs_payloads_json': _n_goal_files,
                  'wrappers': len(_sum_conds), 'conditions': sorted(_sum_conds), 'sums_sha256': PAYLOAD_SUMS_SHA256}
print(f'EMBED OK: {len(EMBED_MANIFEST["files"])} files unpacked to {CODE_DIR} (manifest verified)', flush=True)
print(f'PAYLOAD_SHA256SUMS OK: {_n_files} files + {_n_canon} canonical goal maps; '
      f'{_n_goal_files} goal texts + {len(_sum_conds)} W1 wrappers in payloads.json match; '
      f'sums file sha256 {PAYLOAD_SUMS_SHA256}', flush=True)
sys.path.insert(0, CODE_DIR)
os.environ.pop('BANGLA_PI_PAYLOADS', None)   # attacks must load the verified data/payloads.json

# (d) D41: the stock harness inside this tarball is byte-identical to the main-grid embed that ran
_stock = (_root / 'data/uni_rerun/STOCK_EMBED_MANIFEST.json').read_bytes()
if _sha(_stock) != STOCK_MANIFEST_SHA256:
    raise RuntimeError(f'STOCK_EMBED_MANIFEST sha256 {_sha(_stock)} != {STOCK_MANIFEST_SHA256}')
_stock_bad = [r for r, m in json.loads(_stock)['files'].items()
              if not (_root / r).is_file() or _sha((_root / r).read_bytes()) != m['sha256']]
if _stock_bad:
    raise RuntimeError(f'stock harness files differ from the main-grid embed: {_stock_bad[:5]}')
_new = sorted(set(EMBED_MANIFEST['files']) - set(json.loads(_stock)['files']))
print(f'STOCK HARNESS OK: {len(json.loads(_stock)["files"])} files identical to main-grid embed '
      f'{STOCK_EMBED_SHA256[:12]}…; added: {_new}', flush=True)


In [ ]:
# ===== 2. PARAMETERS (Unicode-preserving rerun arm, D41) =====
MODEL_KEY  = 'gemma'
TIERS      = ['tier_A']          # tier_A = escaped-injection episodes (the arm). Add 'tier_B' only if decided (D41)
RUN_STOCK_RETRY = False      # D35: retry the infrastructure-error episode(s) with the patch OFF
STOCK_RETRY_BUDGET_S = 1500
STREAMS_PER_SERVER = 1           # = main grid (D30/D32): 1 agent stream per llama-server
DEADLINE_H = 11.0                # no NEW episodes after this many hours since cell 1
SMOKE_N    = None                # smoke test only: first N episodes of the arm; None = all
WORK       = '/kaggle/working'
OUTDIR     = '/kaggle/working/runs'
INPUT_ROOT = '/kaggle/input'     # attach a previous version's Output here -> skip-if-done resume
EST_SESSION_H = 0.86   # builder estimate from observed main-grid durations (high 0.97)

from runners.main_grid import MODELS, resolve_model, LLAMA_BUILD, SEED, BENCHMARK_VERSION
from runners import uni_rerun, unicode_patch
MODEL_KEY = resolve_model(MODEL_KEY)
M = MODELS[MODEL_KEY]
LABEL = M['label']               # same label as the stock run; the arm lives in separate files/dirs
DEFENSE = None
RUN_ID_UNI = f'{MODEL_KEY}_150_uni'
RUN_ID_RETRY = f'{MODEL_KEY}_150_stockretry'
_sel = json.loads(Path(f'{CODE_DIR}/data/uni_rerun/SELECTION.json').read_text(encoding='utf-8'))
_ep_path = Path(f'{CODE_DIR}/data/uni_rerun/episodes_{MODEL_KEY}.json')
if _sha(_ep_path.read_bytes()) != _sel['files'][MODEL_KEY]['sha256']:
    raise RuntimeError('episode list sha256 does not match SELECTION.json')
EPISODES = uni_rerun.load_episodes(_ep_path)
SELECTION_SHA256 = _sha(_ep_path.read_bytes())
UNI_TASKS = uni_rerun.arm_tasks(EPISODES, TIERS)
RETRY_TASKS = uni_rerun.retry_tasks(EPISODES) if RUN_STOCK_RETRY else []
_pay = json.loads(Path(f'{CODE_DIR}/data/payloads.json').read_text(encoding='utf-8'))
_filled = lambda v: isinstance(v, str) and v.strip() != '' and not v.strip().startswith('TODO_')
_miss = [t for t in UNI_TASKS + RETRY_TASKS
         if not _filled(_pay['conditions'][t['condition']]['goals'].get(f"{t['suite']}/{t['injection_task']}"))
         or not _filled(_pay['wrappers']['W1'].get(t['condition']))]
if _miss:
    raise RuntimeError(f'{len(_miss)} episodes without payload, e.g. {_miss[:3]}')
assert {t['condition'] for t in UNI_TASKS + RETRY_TASKS} <= set(PAYLOAD_VERIFY['conditions'])
from collections import Counter as _C
print(f'{LABEL}: unicode arm {len(UNI_TASKS)} episodes {dict(_C(t["condition"] for t in UNI_TASKS))} '
      f'(tiers {TIERS}); stock retry {len(RETRY_TASKS)}; selection sha256 {SELECTION_SHA256[:16]}…', flush=True)
print(json.dumps(M, indent=1), flush=True)


In [ ]:
# ===== 3. GPU check (needs Accelerator = GPU T4 x2) =====
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout, flush=True)
_n = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.count('GPU ')
print(f'GPUs visible: {_n}', flush=True)
assert _n >= 2, 'Need Accelerator = GPU T4 x2'


In [ ]:
# ===== 4. Pinned llama.cpp (CUDA 12.8 build) + agentdojo 0.1.35 (same as the pilots) =====
import glob
LLAMA_DIR = '/kaggle/tmp/llama' if os.path.isdir('/kaggle/tmp') else '/tmp/llama'   # keep Output small
os.makedirs(LLAMA_DIR, exist_ok=True)
_rel = f'https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}'
for _url in (f'{_rel}/cudart-llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz',
             f'{_rel}/llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz'):
    _fn = f"{LLAMA_DIR}/{_url.split('/')[-1]}"
    subprocess.run(['wget', '-q', '-O', _fn, _url], check=True)
    subprocess.run(['tar', 'xzf', _fn, '-C', LLAMA_DIR], check=True)
SERVER = glob.glob(f'{LLAMA_DIR}/**/llama-server', recursive=True)[0]
print('llama-server at', SERVER, flush=True)
print(subprocess.run([SERVER, '--version'], capture_output=True, text=True).stderr[:300], flush=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'agentdojo==0.1.35', 'openai==3.23.0'], check=True)


In [ ]:
# ===== 5. Pinned GGUF -> scratch (never /kaggle/working, D13); assert size + SHA-256 =====
from huggingface_hub import hf_hub_download
for _p in glob.glob('/kaggle/working/models/**/*', recursive=True):
    if _p.endswith(('.gguf', '.incomplete', '.lock')) and os.path.isfile(_p):
        os.remove(_p)
MODELS_DIR = '/kaggle/tmp/models' if os.path.isdir('/kaggle/tmp') else '/tmp/models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['HF_HUB_CACHE'] = os.environ['HUGGINGFACE_HUB_CACHE'] = MODELS_DIR
_free = shutil.disk_usage(MODELS_DIR).free / 2**30
print(f'MODELS_DIR={MODELS_DIR} free={_free:.1f} GiB', flush=True)
assert _free > M['min_free_gib'], f"scratch too small for {M['file']}"
_t0 = time.time()
MODEL_PATH = hf_hub_download(M['repo'], M['file'], revision=M['rev'], local_dir=MODELS_DIR)
assert os.path.getsize(MODEL_PATH) == M['size'], f'size mismatch {os.path.getsize(MODEL_PATH)} != {M["size"]}'
_h = hashlib.sha256()
with open(MODEL_PATH, 'rb') as _f:
    while (_b := _f.read(1 << 22)):
        _h.update(_b)
GGUF_SHA = _h.hexdigest()
print(f"{M['file']} {M['size']} B sha256 {GGUF_SHA} match={GGUF_SHA == M['sha256']} ({time.time()-_t0:.0f}s)", flush=True)
assert GGUF_SHA == M['sha256'], 'GGUF sha256 mismatch'


In [ ]:
# ===== 6. Serve in the layout + variant that passed the pilot; prove template kwargs =====
import requests
from runners.main_grid import server_commands, template_ok
PROCS = []
def _wait(p, port, timeout_s=900):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if p.poll() is not None:
            return False
        try:
            if requests.get(f'http://127.0.0.1:{port}/v1/models', timeout=5).ok:
                return True
        except Exception:
            pass
        time.sleep(3)
    return False
def _template_tail(port):
    try:
        r = requests.post(f'http://127.0.0.1:{port}/apply-template', timeout=30,
                          json={'messages': [{'role': 'user', 'content': 'ping'}]})
        return r.json().get('prompt', '')
    except Exception as e:
        return f'<apply-template failed: {e}>'
SPECS = server_commands(M, SERVER, MODEL_PATH, SEED, log_dir=WORK)
for _s in SPECS:
    print(f"serve[{M['variant']}] gpu={_s['gpu']} port={_s['port']}: " + ' '.join(_s['cmd'][1:]), flush=True)
    _t0 = time.time()
    _p = subprocess.Popen(_s['cmd'], env=dict(os.environ, **_s['env']), stdout=open(_s['log'], 'w'),
                          stderr=subprocess.STDOUT)
    PROCS.append(_p)
    if not _wait(_p, _s['port']):
        for _q in PROCS:
            _q.kill()
        raise RuntimeError(f"llama-server on :{_s['port']} failed (rc={_p.poll()}):\n" + open(_s['log']).read()[-2000:])
    print(f"  up on :{_s['port']} ({time.time()-_t0:.0f}s)", flush=True)
PORTS = [_s['port'] for _s in SPECS]
TEMPLATE_TAILS = {p: _template_tail(p) for p in PORTS}
for _p, _t in TEMPLATE_TAILS.items():
    print(f'template[{_p}] tail: {_t[-200:]!r}', flush=True)
TEMPLATE_OK = all(template_ok(M, t) for t in TEMPLATE_TAILS.values())
print(f"TEMPLATE_OK={TEMPLATE_OK} expect={M['template_expect']!r} forbid={M['template_forbid']!r}", flush=True)
if not TEMPLATE_OK:
    for _q in PROCS:
        _q.kill()
    raise RuntimeError('chat-template kwargs did not take effect (thinking/reasoning setting) - not running the grid')
MODEL_ID = requests.get(f'http://127.0.0.1:{PORTS[0]}/v1/models', timeout=10).json()['data'][0]['id']
SERVE_USED = {'layout': M['layout'], 'variant': M['variant'], 'ports': PORTS, 'model_id': MODEL_ID,
              'servers': [{'port': s['port'], 'gpu': s['gpu'], 'cmd': s['cmd'][1:], 'log': s['log']} for s in SPECS],
              'template_ok': TEMPLATE_OK, 'template_tails': {str(k): v[-300:] for k, v in TEMPLATE_TAILS.items()}}
print(f"server up SERVE_VARIANT={M['variant']} layout={M['layout']} ports={PORTS} model_id={MODEL_ID}", flush=True)
print(subprocess.run(['nvidia-smi','--query-gpu=index,memory.used,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout, flush=True)


In [ ]:
# ===== 7. Run: Unicode-preserving arm (patch ON), then the stock retry (patch OFF) =====
unicode_patch.self_check(expect_enabled=False)    # stock behaviour until the arm starts (needs agentdojo, installed in cell 4)
from runners.main_grid import seed_from_inputs
from runners.run_benchmark import PINNED
os.makedirs(OUTDIR, exist_ok=True)
if SMOKE_N:   # smoke test only (never for the real run)
    UNI_TASKS = UNI_TASKS[:SMOKE_N]
    print(f'SMOKE_N={SMOKE_N}: unicode arm limited to {len(UNI_TASKS)} episodes', flush=True)
OUT_UNI, TRACE_UNI = f'{OUTDIR}/{RUN_ID_UNI}.jsonl', f'{OUTDIR}/traces_uni'
OUT_RETRY, TRACE_RETRY = f'{OUTDIR}/{RUN_ID_RETRY}.jsonl', f'{OUTDIR}/traces_stockretry'
SEEDED = {}
for _rid, _out, _tr in ((RUN_ID_UNI, OUT_UNI, TRACE_UNI), (RUN_ID_RETRY, OUT_RETRY, TRACE_RETRY)):
    if not os.path.exists(_out):
        SEEDED[_rid] = {k: (v if not isinstance(v, list) else len(v))
                        for k, v in seed_from_inputs(INPUT_ROOT, _out, _tr, _rid).items()}
print('resume seed from inputs:', SEEDED, flush=True)
HEADER = {'model_key': MODEL_KEY, 'subset': '150', 'serve': SERVE_USED, 'gguf_sha256': GGUF_SHA,
          'llama_build': LLAMA_BUILD, 'seed': SEED, 'pinned': PINNED, 'embed_sha256': EMBED_SHA256,
          'stock_embed_sha256': STOCK_EMBED_SHA256, 'payload_sums_sha256': PAYLOAD_SUMS_SHA256,
          'selection_sha256': SELECTION_SHA256, 'tiers': TIERS, 'decision': 'D41'}
DEADLINE_TS = SESSION_T0 + DEADLINE_H * 3600
STATS_UNI = uni_rerun.run_arm(UNI_TASKS, PORTS, unicode=True, label=LABEL, out_jsonl=OUT_UNI, trace_dir=TRACE_UNI,
                              header=dict(HEADER, run_id=RUN_ID_UNI), deadline_ts=DEADLINE_TS,
                              streams_per_port=STREAMS_PER_SERVER)
print('STATS_UNI', json.dumps({k: v for k, v in STATS_UNI.items() if k != 'worker_errors'}), flush=True)
STATS_RETRY = {}
if RETRY_TASKS and time.time() < DEADLINE_TS:
    STATS_RETRY = uni_rerun.run_stock_retry(RETRY_TASKS, PORTS, label=LABEL, out_jsonl=OUT_RETRY,
                                            trace_dir=TRACE_RETRY, header=dict(HEADER, run_id=RUN_ID_RETRY),
                                            budget_s=STOCK_RETRY_BUDGET_S)
    print('STATS_RETRY', json.dumps({k: v for k, v in STATS_RETRY.items() if k != 'worker_errors'}, default=str), flush=True)
for _s in (STATS_UNI, STATS_RETRY):
    if _s.get('worker_errors'):
        print('WORKER ERRORS:', _s['worker_errors'], flush=True)


In [ ]:
# ===== 8. Stop servers, manifests, zipped traces, UNI_RESULT / STOCK_RETRY_RESULT lines =====
import datetime as _dt
from runners.main_grid import summarize, zip_traces, main_result_line
for _p in PROCS:
    try:
        _p.terminate()
    except Exception:
        pass
WALL_S = time.time() - SESSION_T0
RESULT_LINES = []
for _rid, _arm, _tasks, _out, _tr, _st, _tag in (
        (RUN_ID_UNI, 'unicode_preserving', UNI_TASKS, OUT_UNI, TRACE_UNI, STATS_UNI, 'UNI_RESULT'),
        (RUN_ID_RETRY, 'stock_retry', RETRY_TASKS, OUT_RETRY, TRACE_RETRY, STATS_RETRY, 'STOCK_RETRY_RESULT')):
    if not _tasks:
        continue
    _sum = summarize(_out, _tasks)
    _zip = zip_traces(_tr, f'{WORK}/{_rid}_traces.zip')
    _man = {'run_id': _rid, 'arm': _arm, 'decision': 'D41', 'model_key': MODEL_KEY, 'label': LABEL,
            'tiers': TIERS if _arm == 'unicode_preserving' else None, 'n_tasks': len(_tasks),
            'streams_per_server': STREAMS_PER_SERVER, 'smoke_n': SMOKE_N,
            'gguf': {k: M[k] for k in ('repo', 'rev', 'file', 'size')} | {'sha256': GGUF_SHA},
            'serve': SERVE_USED, 'llama_build': LLAMA_BUILD, 'seed': SEED, 'pinned': PINNED,
            'embed': {'sha256': EMBED_SHA256, 'bytes': EMBED_BYTES, 'n_files': EMBED_N_FILES},
            'stock_embed_sha256': STOCK_EMBED_SHA256, 'selection_sha256': SELECTION_SHA256,
            'unicode_patch': unicode_patch.describe() | {'enabled_for_arm': _arm == 'unicode_preserving'},
            'payload_verify': PAYLOAD_VERIFY, 'resume_seeded': SEEDED.get(_rid),
            'stats': _st, 'summary': _sum, 'traces_zip': _zip, 'est_session_h': EST_SESSION_H,
            'session_started': _dt.datetime.fromtimestamp(SESSION_T0).isoformat(timespec='seconds'),
            'session_ended': _dt.datetime.now().isoformat(timespec='seconds'), 'wall_s': round(WALL_S, 1),
            'outputs': {'jsonl': _out, 'traces_dir': _tr, 'traces_zip': _zip['zip'],
                        'server_logs': [s['log'] for s in SERVE_USED['servers']]}}
    Path(f'{WORK}/runs_manifest_{_rid}.json').write_text(json.dumps(_man, indent=2, ensure_ascii=False, default=str),
                                                         encoding='utf-8')
    print(f'wrote runs_manifest_{_rid}.json, {_out}, {_zip["zip"]} ({_zip["n_traces"]} traces)', flush=True)
    if not _sum['complete']:
        print(f"INCOMPLETE {_rid}: {_sum['missing']} missing + {_sum['errors']} errored. To resume, attach this "
              f"version's Output as an input and Save & Run All again (skip-if-done).", flush=True)
    RESULT_LINES.append(main_result_line(_rid, LABEL, '1/1', _sum, WALL_S, _st, EMBED_SHA256)
                        .replace('MAIN_RESULT ', _tag + ' ', 1))
for _l in RESULT_LINES:
    print(_l, flush=True)


## After the run (on the box)
Download the Output into `code/results/uni/<model>/`, check the `UNI_RESULT` line and that `embed_sha256` matches
the header, then `python -m analysis.unicode_arm …` (see `analysis/README.md`, D41). The stock retry
(`*_stockretry.jsonl` + traces) is folded into the primary set with `run_analysis --retry-scores/--retry-traces`.
Record slug/version, wall time and the result lines in `DECISIONS.md`.